In [26]:
import pandas as pd
import numpy as np
import plotly.express as px

# =========================
# Preparar datos
# =========================
df_matrix = dataset_limpo_medianag[['DevType', 'YearsCodePro', 'ConvertedCompYearly']].copy()

df_matrix['YearsCodePro'] = pd.to_numeric(df_matrix['YearsCodePro'], errors='coerce')
df_matrix['ConvertedCompYearly'] = pd.to_numeric(df_matrix['ConvertedCompYearly'], errors='coerce')

df_matrix = df_matrix[
    (df_matrix['ConvertedCompYearly'] > 1000) & 
    (df_matrix['ConvertedCompYearly'] < 500000)
].copy()

USD_TO_EUR = 0.92
df_matrix['SalaryEUR'] = df_matrix['ConvertedCompYearly'] * USD_TO_EUR

# =========================
# Experiencia
# =========================
exp_bins = [-0.001, 1, 3, 5, 8, 12, 20, np.inf]
exp_labels = ['0-1y', '2-3y', '4-5y', '6-8y', '9-12y', '13-20y', '21+y']

df_matrix['ExpBand'] = pd.cut(
    df_matrix['YearsCodePro'],
    bins=exp_bins,
    labels=exp_labels,
    include_lowest=True
)

# =========================
# Top roles
# =========================
devtypes_clean = (
    df_matrix['DevType']
    .astype(str)
    .str.split(';')
    .explode()
    .str.strip()
)

devtypes_clean = devtypes_clean[
    (devtypes_clean != '') &
    (devtypes_clean != 'Sem dado') &
    (devtypes_clean != 'Sem dados') &
    (devtypes_clean != 'Other (please specify):')
]

top_devtypes = devtypes_clean.value_counts().head(6).index.tolist()

# =========================
# Rol principal
# =========================
def get_primary_role(devtype_str, top_roles):
    for role in top_roles:
        if str(role).lower() in str(devtype_str).lower():
            return role
    return 'Other'

df_matrix['PrimaryRole'] = df_matrix['DevType'].astype(str).apply(
    lambda x: get_primary_role(x, top_devtypes)
)

df_matrix = df_matrix[df_matrix['PrimaryRole'] != 'Other'].copy()

# =========================
# Crear matriz
# =========================
matrix_data = []

for role in top_devtypes:
    row = []
    for exp in exp_labels:
        subset = df_matrix[
            (df_matrix['PrimaryRole'] == role) & 
            (df_matrix['ExpBand'] == exp)
        ]
        avg_salary = subset['SalaryEUR'].mean() if len(subset) > 10 else np.nan
        row.append(avg_salary)
    matrix_data.append(row)

salary_matrix = pd.DataFrame(
    matrix_data,
    index=top_devtypes,
    columns=exp_labels
)

# =========================
# Heatmap Plotly
# =========================
fig = px.imshow(
    salary_matrix,
    text_auto=".0f",
    color_continuous_scale="RdYlGn",
    aspect="auto",
    labels=dict(x="Years of Experience", y="Developer Role", color="Salary (€)"),
)

# Formato de texto (€)
fig.update_traces(
    texttemplate="€%{z:.0f}",
    hovertemplate="Role: %{y}<br>Experience: %{x}<br>Salary: €%{z:,.0f}<extra></extra>"
)

fig.update_layout(
    template="plotly_white",
    height=600
)

# =========================
# Exportar + mostrar
# =========================
fig.write_html('./Public/assets/career_pathw3.html')
fig.show()

print("✓ Matrix exported successfully")

✓ Matrix exported successfully
